# NYT Stock Headlines Crawler

1990~2025 NYT "stock" 기사 헤드라인 수집
- `sort=relevance`: 월별 관련도 상위 50건
- 체크포인트: 끊겨도 셀 재실행하면 이어서 수집
- 결과: Google Drive에 CSV 저장

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/Colab Notebooks/nynews'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Save directory: {SAVE_DIR}')

# 기존 데이터 확인
for f in ['nyt_headlines.csv', 'crawl_checkpoint.json']:
    path = os.path.join(SAVE_DIR, f)
    if os.path.exists(path):
        size = os.path.getsize(path)
        print(f'  {f}: {size:,} bytes')
    else:
        print(f'  {f}: not found')

In [ ]:
API_KEY = 'V21sbXOFUnMAoigNw0YRlCrxahLKbKzToiamQpkEoECGMpiX'

In [ ]:
import requests
import time
import csv
import json
import calendar
from datetime import datetime

BASE_URL = 'https://api.nytimes.com/svc/search/v2/articlesearch.json'
QUERY = 'stock'
START_YEAR = 1990
END_YEAR = 2025

REQUEST_DELAY = 15       # seconds between requests (safe margin)
RATE_LIMIT_WAIT = 120    # seconds on 429 (longer cooldown)
EMPTY_RESULT_WAIT = 60   # seconds on empty result
MAX_RETRIES = 5          # more retries before giving up
MAX_PAGES = 5            # pages per month (5 pages = 50 articles)

CSV_FILE = os.path.join(SAVE_DIR, 'nyt_headlines.csv')
CHECKPOINT_FILE = os.path.join(SAVE_DIR, 'crawl_checkpoint.json')

CSV_COLUMNS = [
    'date', 'year_month', 'headline', 'abstract', 'lead_paragraph',
    'word_count', 'section', 'news_desk', 'type_of_material', 'keywords', 'web_url',
]


def load_checkpoint():
    if not os.path.exists(CHECKPOINT_FILE):
        return {'completed_months': [], 'current_month': None, 'current_page': 0}
    with open(CHECKPOINT_FILE, 'r') as f:
        return json.load(f)


def save_checkpoint(cp):
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(cp, f, indent=2)


def init_csv():
    header_line = ','.join(CSV_COLUMNS)
    if not os.path.exists(CSV_FILE):
        with open(CSV_FILE, 'w', encoding='utf-8', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
            writer.writeheader()
    else:
        with open(CSV_FILE, 'r', encoding='utf-8') as f:
            first_line = f.readline().strip()
        if first_line != header_line:
            with open(CSV_FILE, 'r', encoding='utf-8') as f:
                content = f.read()
            with open(CSV_FILE, 'w', encoding='utf-8', newline='') as f:
                f.write(header_line + '\n' + content)


def append_rows(rows):
    with open(CSV_FILE, 'a', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
        writer.writerows(rows)


def extract_keywords(doc):
    return '; '.join(kw.get('value', '') for kw in doc.get('keywords', []) if kw.get('value'))


def parse_doc(doc, year_month):
    return {
        'date': doc.get('pub_date', ''),
        'year_month': year_month,
        'headline': doc.get('headline', {}).get('main', ''),
        'abstract': doc.get('abstract', ''),
        'lead_paragraph': doc.get('lead_paragraph', ''),
        'word_count': doc.get('word_count', 0),
        'section': doc.get('section_name', ''),
        'news_desk': doc.get('news_desk', ''),
        'type_of_material': doc.get('type_of_material', ''),
        'keywords': extract_keywords(doc),
        'web_url': doc.get('web_url', ''),
    }


def fetch_page(begin_date, end_date, page):
    params = {
        'q': QUERY,
        'begin_date': begin_date,
        'end_date': end_date,
        'api-key': API_KEY,
        'page': page,
        'sort': 'relevance',
    }
    resp = requests.get(BASE_URL, params=params, timeout=30)
    if resp.status_code == 429:
        return 'rate_limit'
    if resp.status_code == 401:
        raise RuntimeError('401 Unauthorized - check API key')
    resp.raise_for_status()
    docs = resp.json().get('response', {}).get('docs')
    return docs if docs is not None else []


def fetch_month(year, month, start_page, checkpoint):
    _, last_day = calendar.monthrange(year, month)
    begin = f'{year}{month:02d}01'
    end = f'{year}{month:02d}{last_day:02d}'
    ym = f'{year}-{month:02d}'

    page = start_page
    total = 0
    consecutive_empty = 0

    while page < MAX_PAGES:
        retries = 0
        docs = None
        while retries < MAX_RETRIES:
            try:
                result = fetch_page(begin, end, page)
                if result == 'rate_limit':
                    wait = RATE_LIMIT_WAIT * (retries + 1)  # exponential backoff
                    print(f'    [429] page {page} - {wait}s wait... (retry {retries+1}/{MAX_RETRIES})')
                    time.sleep(wait)
                    retries += 1
                    continue
                docs = result
                break
            except Exception as e:
                print(f'    [error] page {page}: {e}')
                time.sleep(30)
                retries += 1

        if docs is None:
            page += 1
            continue

        if not docs:
            consecutive_empty += 1
            if consecutive_empty == 1:
                time.sleep(EMPTY_RESULT_WAIT)
                continue
            elif consecutive_empty >= 3:
                break
            else:
                page += 1
                time.sleep(REQUEST_DELAY)
                continue

        consecutive_empty = 0
        rows = [parse_doc(d, ym) for d in docs]
        append_rows(rows)
        total += len(rows)
        checkpoint['current_page'] = page + 1
        save_checkpoint(checkpoint)
        page += 1
        time.sleep(REQUEST_DELAY)

    return total


# API test
print('Testing API...', end=' ')
time.sleep(3)
try:
    test = fetch_page('20200101', '20200131', 0)
    if test == 'rate_limit':
        print('429 - API still rate limited. Wait a few minutes and retry.')
    elif isinstance(test, list):
        print(f'OK! Got {len(test)} docs.')
except Exception as e:
    print(f'Error: {e}')

In [ ]:
# ── Run Crawler ──
# 끊기면 이 셀만 다시 실행하면 이어서 수집

init_csv()
checkpoint = load_checkpoint()
completed = set(checkpoint['completed_months'])
print(f'Completed: {len(completed)} months')
print(f'Range: {START_YEAR}-01 ~ {END_YEAR}-12')
print(f'Remaining: ~{(END_YEAR - START_YEAR + 1) * 12 - len(completed)} months')
print()

now = datetime.now()
grand_total = 0

for year in range(START_YEAR, END_YEAR + 1):
    for month in range(1, 13):
        if datetime(year, month, 1) > now:
            break

        ym = f'{year}-{month:02d}'
        if ym in completed:
            continue

        start_page = 0
        if checkpoint.get('current_month') == ym:
            start_page = checkpoint.get('current_page', 0)

        print(f'[{ym}]', end=' ')

        checkpoint['current_month'] = ym
        checkpoint['current_page'] = start_page
        save_checkpoint(checkpoint)

        count = fetch_month(year, month, start_page, checkpoint)
        grand_total += count
        print(f'{count} articles (total: {grand_total})')

        checkpoint['completed_months'].append(ym)
        checkpoint['current_month'] = None
        checkpoint['current_page'] = 0
        save_checkpoint(checkpoint)
        completed.add(ym)

print(f'\nDone! {grand_total} new articles')
print(f'File: {CSV_FILE}')

In [ ]:
# ── Check Result ──
import pandas as pd

df = pd.read_csv(CSV_FILE)
print(f'Total: {len(df)} articles')
print(f'Date range: {df["date"].min()[:10]} ~ {df["date"].max()[:10]}')
print(f'Months: {df["year_month"].nunique()}')
print()
print(df['year_month'].value_counts().sort_index().head(20))